In [1]:
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from tqdm import tqdm
import os

from tensorflow.keras.preprocessing.sequence import TimeseriesGenerator
from sklearn.metrics import accuracy_score, classification_report, log_loss

In [2]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [3]:
DATASET_PATH = Path("/content/drive/MyDrive/Colab Notebooks/tdcsfog")
TRAIN_PATH = DATASET_PATH / "train"
TEST_PATH = DATASET_PATH / "test"


In [4]:
def load_file(file_path: Path) -> tuple[np.ndarray, float]:

    # Load the data into a numpy ndarray
    with open(file_path, "rb") as infile:
        arr = np.load(infile)

    # Extract the accelerometer data as the input features
    features = arr[:, 1:4]

    # Extract the labels
    labels = arr[:, 4:]
    labels = np.max(labels, axis=-1)
    return features, np.any(labels).astype(float)

In [5]:
train_files = [f for f in TRAIN_PATH.iterdir() if f.is_file() and f.suffix == ".npy"]

train_df = {"file_name": [], "label": []}
X_train = np.zeros(shape=(len(train_files), 3, 1280))
y_train = np.zeros(shape=(len(train_files),))
for idx, train_file in enumerate(tqdm(train_files, desc="Processing files")):
    # Get features and corresponding label
    features, label = load_file(train_file)
    X_train[idx, :, :] = features.T  # note we transpote features array
    y_train[idx] = label

    # Bookkeeping
    train_df["file_name"].append(train_file.name)
    train_df["label"].append(label)
train_df = pd.DataFrame(train_df)
if not DATASET_PATH.joinpath("train.csv").exists():
    train_df.to_csv(DATASET_PATH / "train.csv", index=False, header=True)

Processing files: 100%|██████████| 4136/4136 [01:42<00:00, 40.24it/s] 


In [6]:
test_files = [f for f in TEST_PATH.iterdir() if f.is_file() and f.suffix == ".npy"]

test_df = {"file_name": [], "label": []}
X_test = np.zeros(shape=(len(test_files), 3, 1280))
y_test = np.zeros(shape=(len(test_files),))
for idx, test_file in enumerate(tqdm(test_files, desc="Processing files")):
    # Get features and corresponding label
    features, label = load_file(test_file)
    X_test[idx, :, :] = features.T  # note we transpote features array
    y_test[idx] = label

    # Bookkeeping
    test_df["file_name"].append(test_file.name)
    test_df["label"].append(label)
test_df = pd.DataFrame(test_df)
if not DATASET_PATH.joinpath("test.csv").exists():
    test_df.to_csv(DATASET_PATH / "test.csv", index=False, header=True)

Processing files: 100%|██████████| 960/960 [00:18<00:00, 52.94it/s] 


In [7]:
X_train.shape, y_train.shape, X_test.shape, y_test.shape

((4136, 3, 1280), (4136,), (960, 3, 1280), (960,))

In [10]:
X_train_seq = X_train.transpose(0, 2, 1)
X_train_seq.shape

X_test_seq = X_test.transpose(0, 2, 1)
X_test_seq.shape

(960, 1280, 3)

In [25]:
import numpy as np

generator =TimeseriesGenerator(X_train_seq, y_train, length=1280, batch_size=128)

In [28]:
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import LSTM, Dense, Dropout

model = Sequential([
    LSTM(64, input_shape=(1280, 3), return_sequences=False),
    Dropout(0.3),
    Dense(32, activation='relu'),
    Dense(1, activation='sigmoid')  # Or 'softmax' if multi-class
])

model.summary()


/usr/local/lib/python3.11/dist-packages/keras/src/layers/rnn/rnn.py:200: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)


Model: "sequential_3"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ lstm (LSTM)                     │ (None, 64)             │        17,408 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_3 (Dropout)             │ (None, 64)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_6 (Dense)                 │ (None, 32)             │         2,080 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_7 (Dense)                 │ (None, 1)              │            33 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 19,521 (76.25 KB)

 Trainable params: 19,521 (76.25 KB)

 Non-trainable params: 0 (0.00 B)

In [29]:
model.compile(optimizer='adam',
              loss='binary_crossentropy',  # change if multi-class
              metrics=['accuracy'])

history= model.fit(X_train_seq, y_train, validation_data=(X_test_seq, y_test), epochs=10, batch_size=32)


Epoch 1/10
130/130 ━━━━━━━━━━━━━━━━━━━━ 71s 530ms/step - accuracy: 0.5927 - loss: 0.6597 - val_accuracy: 0.5656 - val_loss: 0.6425
Epoch 2/10
 47/130 ━━━━━━━━━━━━━━━━━━━━ 38s 464ms/step - accuracy: 0.5990 - loss: 0.6379

KeyboardInterrupt: 